1. Caricamento training.csv

2. Preparazione dei dati

3. Train/Test Split

4. Baseline Model

5. Addestramento di più classificatori

6. Confronto dei risultati

7. Cross Validation

8. Ottimizzazione degli iperparametri

9. Analisi Bias-Variance

10. Selezione del modello finale

11. Conclusioni

# 1. Caricamento del dataset

In questo notebook vengono addestrati e confrontati diversi classificatori mediante la libreria Scikit-Learn.

L'obiettivo è individuare il modello che ottiene le migliori prestazioni sul test set e che verrà successivamente utilizzato sul file `real_settings.csv` fornito in sede d'esame.

In [5]:
import pandas as pd

training_df = pd.read_csv("../data/training.csv")

training_df.shape

(50000, 22)

# 2. Preparazione dei dati

Per il Task 5 vengono utilizzate tutte le feature disponibili nel dataset.

A differenza del Task 2, non è necessario effettuare discretizzazioni manuali poiché gli algoritmi di Scikit-Learn sono in grado di gestire direttamente variabili numeriche e categoriche codificate come interi.

In [6]:
X = training_df.drop(columns=["Diabetes_binary"])

y = training_df["Diabetes_binary"]

print(X.shape)
print(y.shape)

(50000, 21)
(50000,)


# 3. Suddivisione in Training Set e Test Set

Il dataset viene suddiviso in training set e test set.

Viene utilizzata una suddivisione stratificata per preservare la distribuzione delle classi presente nel dataset originale.

Questo approccio è particolarmente importante in presenza di dataset sbilanciati.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(40000, 21)
(10000, 21)


# 4. Baseline Model

Come riferimento iniziale viene considerato un classificatore banale che predice sempre la classe maggioritaria.

Questo permette di verificare se i modelli addestrati apportano un reale miglioramento.

In [8]:
y.value_counts(normalize=True) * 100

Diabetes_binary
0    86.066
1    13.934
Name: proportion, dtype: float64

## 5.0 Preparazione tabella risultati

Ogni classificatore aggiungerà una riga.

5. Addestramento dei classificatori

5.1 Naive Bayes

5.2 Decision Tree

5.3 Logistic Regression

5.4 k-NN

5.5 Perceptron

6. Confronto dei risultati

7. Cross Validation

8. Ottimizzazione degli iperparametri

9. Selezione del modello finale

In [22]:
results_table = []

## 5.1 Naive Bayes

Il classificatore Naive Bayes assume indipendenza condizionata tra le feature dato il valore della classe.

Nonostante tale ipotesi sia spesso irrealistica, il modello risulta molto efficiente e rappresenta uno dei principali algoritmi probabilistici studiati durante il corso.

In [9]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)

## Metriche

- Accuracy
- Precision
- Recall
- F1-Score

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


In [25]:
results_table.append({
    "Model": "Naive Bayes",
    "Accuracy": accuracy_score(y_test, nb_pred),
    "Precision": precision_score(y_test, nb_pred),
    "Recall": recall_score(y_test, nb_pred),
    "F1": f1_score(y_test, nb_pred)
})

In [27]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956


### Discussione dei risultati

Il classificatore Naive Bayes ottiene un'accuracy pari al 77.98%.

Tale valore appare inizialmente elevato, ma deve essere interpretato alla luce del forte sbilanciamento presente nel dataset. Infatti la classe negativa rappresenta circa l'86% delle osservazioni e un classificatore banale che predicesse sempre la classe maggioritaria raggiungerebbe un'accuracy superiore.

Per questo motivo risultano particolarmente importanti le metriche precision, recall e F1-score.

La recall pari al 57.93% indica che il modello riesce a identificare oltre la metà dei soggetti diabetici presenti nel test set.

La precision pari al 33.31% evidenzia invece la presenza di numerosi falsi positivi.

Nel complesso il modello rappresenta una buona baseline probabilistica, ma lascia ampio margine di miglioramento.

## 5.2 Decision Tree

Gli alberi decisionali sono modelli di classificazione che suddividono progressivamente il dataset in sottoinsiemi sempre più omogenei rispetto alla variabile target.

Ad ogni nodo viene selezionata la feature che massimizza la separazione tra le classi secondo una misura di impurità.

Nel caso di Scikit-Learn viene utilizzato l'indice di Gini come criterio predefinito.

Uno dei principali vantaggi degli alberi decisionali è la loro elevata interpretabilità: il processo decisionale può essere rappresentato graficamente e tradotto in un insieme di regole IF-THEN.

In [10]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(X_train, y_train) 

dt_pred = dt.predict(X_test)

In [29]:
results_table.append({
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_test, dt_pred),
    "Precision": precision_score(y_test, dt_pred),
    "Recall": recall_score(y_test, dt_pred),
    "F1": f1_score(y_test, dt_pred)
})

In [30]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972


### Confronto preliminare con Naive Bayes

Il Decision Tree ottiene un'accuracy leggermente superiore rispetto al Naive Bayes.

Tuttavia il dataset presenta un forte sbilanciamento delle classi e, in questo contesto, l'accuracy non rappresenta la metrica più significativa.

Analizzando recall e F1-score emerge che il Naive Bayes risulta attualmente più efficace nell'identificare i soggetti diabetici.

In particolare:

- il Decision Tree raggiunge una recall pari al 32.74%;
- il Naive Bayes raggiunge una recall pari al 57.93%.

Anche l'F1-score risulta superiore per il Naive Bayes.

Pertanto, nonostante la maggiore accuracy, il Decision Tree non può ancora essere considerato il modello migliore.

## 5.3 Logistic Regression

La regressione logistica è un modello lineare per la classificazione.

A differenza della regressione lineare, l'obiettivo non è predire un valore numerico continuo ma la probabilità di appartenenza ad una determinata classe.

Il modello utilizza la funzione logistica (sigmoide):

\[
\sigma(z)=\frac{1}{1+e^{-z}}
\]

che trasforma qualsiasi valore reale in una probabilità compresa tra 0 e 1.

Nel caso della classificazione binaria, la classe viene assegnata confrontando la probabilità stimata con una soglia decisionale (generalmente pari a 0.5).

Poiché la regressione logistica è sensibile alle diverse scale delle feature, prima dell'addestramento viene effettuata una standardizzazione dei dati.

### Standardizzazione delle feature

Le feature del dataset presentano scale molto differenti.

Ad esempio:

- BMI varia circa tra 12 e 98;
- Age varia tra 1 e 13;
- Income varia tra 1 e 8;
- MentHlth e PhysHlth variano tra 0 e 30.

Per evitare che le variabili con valori numerici più elevati dominino il processo di apprendimento, viene applicata una standardizzazione.

La standardizzazione trasforma ogni feature in modo da avere:

\[
\mu = 0
\]

\[
\sigma = 1
\]

dove \(\mu\) rappresenta la media e \(\sigma\) la deviazione standard.

In [13]:
from sklearn.preprocessing import StandardScaler 

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [32]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)

In [34]:
results_table.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, lr_pred),
    "Precision": precision_score(y_test, lr_pred),
    "Recall": recall_score(y_test, lr_pred),
    "F1": f1_score(y_test, lr_pred)
})

In [35]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972
2,Logistic Regression,0.8635,0.535000,0.153625,0.238706


### Analisi dei risultati

La Logistic Regression ottiene l'accuracy più elevata tra i modelli analizzati fino a questo momento (86.35%).

Tuttavia, il dataset presenta un forte sbilanciamento delle classi: circa l'86% delle osservazioni appartiene alla classe "non diabetico". In questo contesto una elevata accuracy non implica necessariamente una buona capacità di individuare i soggetti diabetici.

Infatti la recall risulta pari ad appena il 15.36%, valore significativamente inferiore rispetto a quello ottenuto dal Naive Bayes (57.93%).

Questo significa che il modello tende a classificare la maggior parte delle osservazioni come non diabetiche, riducendo il numero di falsi positivi ma aumentando notevolmente il numero di falsi negativi.

La precision (53.50%) è invece la più elevata tra i modelli analizzati: quando il modello predice la presenza di diabete, la previsione risulta spesso corretta.

Dal punto di vista pratico, il modello privilegia la precision rispetto alla recall. In un contesto sanitario ciò potrebbe non essere ideale, poiché perdere molti casi reali di diabete può essere più grave che produrre alcuni falsi allarmi.

Anche l'F1-score (23.87%) risulta inferiore a quello del Naive Bayes, indicando che il compromesso complessivo tra precision e recall non è ancora ottimale.

## 5.4 k-Nearest Neighbors (k-NN)

L'algoritmo k-NN appartiene alla categoria dei metodi Instance-Based Learning.

Per classificare una nuova osservazione, il modello identifica i k campioni più vicini nel training set e assegna la classe più frequente tra essi.

La vicinanza tra le osservazioni viene calcolata mediante una misura di distanza, generalmente la distanza euclidea.

Poiché il calcolo delle distanze è fortemente influenzato dalla scala delle feature, il modello viene addestrato utilizzando i dati standardizzati.

In [36]:
from sklearn.neighbors import KNeighborsClassifier

k = 5

knn = KNeighborsClassifier(
    n_neighbors=5
)

knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_test_scaled)

In [37]:
results_table.append({
    "Model": "k-NN (k=5)",
    "Accuracy": accuracy_score(y_test, knn_pred),
    "Precision": precision_score(y_test, knn_pred),
    "Recall": recall_score(y_test, knn_pred),
    "F1": f1_score(y_test, knn_pred)
})

In [38]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972
2,Logistic Regression,0.8635,0.535000,0.153625,0.238706
3,k-NN (k=5),0.8519,0.434524,0.209620,0.282809


### Analisi dei risultati

Il classificatore k-NN ottiene un'accuracy pari all'85.19%, valore inferiore alla Logistic Regression ma superiore a Naive Bayes e Decision Tree.

La precision (43.45%) risulta relativamente elevata, indicando che una parte consistente delle osservazioni classificate come diabetiche appartiene effettivamente alla classe positiva.

Tuttavia la recall (20.96%) è piuttosto bassa: il modello riesce a individuare soltanto circa un quinto dei soggetti diabetici presenti nel test set.

Questo comportamento è tipico dei dataset sbilanciati. Poiché la maggior parte dei vicini appartiene alla classe non diabetica, il modello tende a favorire la classe maggioritaria.

L'F1-score (28.28%) risulta superiore a quello della Logistic Regression ma inferiore sia al Decision Tree sia al Naive Bayes.

Nel complesso il modello mostra buone capacità di classificazione generale ma una limitata sensibilità nell'identificazione dei casi di diabete.

## 5.5 Perceptron

Il Perceptron è uno dei più semplici algoritmi di apprendimento supervisionato per la classificazione binaria.

Il modello apprende iterativamente un iperpiano di separazione aggiornando i pesi delle feature quando viene commesso un errore di classificazione.

Dal punto di vista teorico rappresenta uno dei primi algoritmi di Machine Learning sviluppati per la classificazione lineare.

Poiché il processo di apprendimento dipende direttamente dai valori numerici delle feature, il modello viene addestrato utilizzando i dati standardizzati.

In [ ]:
from sklearn.linear_model import Perceptron

perceptron = Perceptron(
    random_state=42,
    max_iter=1000
)

perceptron.fit(X_train_scaled, y_train)

perc_pred = perceptron.predict(X_test_scaled)

In [41]:
results_table.append({
    "Model": "Perceptron",
    "Accuracy": accuracy_score(y_test, perc_pred),
    "Precision": precision_score(y_test, perc_pred),
    "Recall": recall_score(y_test, perc_pred),
    "F1": f1_score(y_test, perc_pred)
})

In [42]:
pd.DataFrame(results_table)

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972
2,Logistic Regression,0.8635,0.535000,0.153625,0.238706
3,k-NN (k=5),0.8519,0.434524,0.209620,0.282809
4,Perceptron,0.7898,0.297313,0.373295,0.330999


### Analisi dei risultati

Il Perceptron ottiene un'accuracy pari al 78.98%, un valore inferiore rispetto a Logistic Regression, k-NN e Decision Tree.

La precision (29.73%) risulta relativamente bassa, indicando che una parte significativa delle osservazioni classificate come diabetiche appartiene in realtà alla classe negativa.

La recall (37.33%) è invece superiore a quella ottenuta da Logistic Regression (15.36%), k-NN (20.96%) e Decision Tree (32.74%), ma inferiore rispetto al Naive Bayes (57.93%).

L'F1-score raggiunge il valore di 33.10%, collocandosi tra il Naive Bayes e gli altri modelli considerati.

Dal punto di vista teorico, il risultato è coerente con la natura del Perceptron: essendo un classificatore lineare, è in grado di apprendere soltanto frontiere decisionali lineari. Se il problema presenta relazioni più complesse tra le feature, il modello può risultare limitato rispetto ad approcci più flessibili.

Nel complesso il Perceptron mostra prestazioni discrete ma non rappresenta il modello migliore tra quelli analizzati.

# 6. Confronto dei risultati

In questa sezione vengono confrontate le prestazioni dei diversi classificatori addestrati sul dataset.

Poiché il dataset presenta un forte sbilanciamento delle classi, non è sufficiente considerare esclusivamente l'accuracy.

Per questo motivo vengono analizzate anche precision, recall e F1-score, che consentono di valutare più correttamente la capacità dei modelli di identificare i soggetti diabetici.

In [43]:
results_df = pd.DataFrame(results_table)

results_df

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.7798,0.333058,0.579325,0.422956
1,Decision Tree,0.7998,0.299803,0.327351,0.312972
2,Logistic Regression,0.8635,0.535000,0.153625,0.238706
3,k-NN (k=5),0.8519,0.434524,0.209620,0.282809
4,Perceptron,0.7898,0.297313,0.373295,0.330999


## Discussione critica dei risultati

I risultati mostrano comportamenti differenti tra i classificatori analizzati.

La Logistic Regression ottiene la massima accuracy (86.35%) e la massima precision (53.50%), ma presenta una recall molto bassa (15.36%). Ciò significa che il modello tende a classificare la maggior parte delle osservazioni come non diabetiche, perdendo numerosi casi reali di diabete.

Il Naive Bayes ottiene invece la migliore recall (57.93%) e il miglior F1-score (42.30%). Questo indica una maggiore capacità di individuare i soggetti diabetici, mantenendo un equilibrio migliore tra precision e recall.

Decision Tree e Perceptron mostrano prestazioni intermedie, mentre il k-NN risulta penalizzato dallo sbilanciamento delle classi.

Considerando che il problema affrontato riguarda la diagnosi del diabete, la capacità di identificare correttamente i soggetti positivi assume particolare importanza. Per questo motivo il Naive Bayes rappresenta attualmente il modello più promettente tra quelli analizzati.

# 7. Cross Validation

Le prestazioni ottenute nella fase precedente sono state calcolate utilizzando un singolo train/test split.

Tale procedura può produrre risultati influenzati dalla particolare suddivisione dei dati scelta.

Per ottenere una stima più robusta delle prestazioni dei classificatori viene utilizzata la tecnica della Cross Validation.

In particolare si utilizza una Stratified 10-Fold Cross Validation, che suddivide il dataset in 10 parti mantenendo invariata la distribuzione delle classi in ciascun fold.

Poiché il dataset è fortemente sbilanciato (circa 86% non diabetici e 14% diabetici), è opportuno utilizzare una versione stratificata della Cross Validation.

La stratificazione garantisce che ogni fold mantenga una distribuzione delle classi simile a quella presente nell'intero dataset.

Ad ogni iterazione:

- 9 fold vengono utilizzati per l'addestramento;
- 1 fold viene utilizzato per la validazione;

Il processo viene ripetuto 10 volte e le metriche finali vengono ottenute come media dei risultati.

Validiamo:

- Naive Bayes
- Decision Tree
- Logistic Regression

In [1]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

# Creazione dello splitter
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)


In [11]:
# Naive Bayes
nb_scores = cross_val_score(
    GaussianNB(),
    X,
    y,
    cv=cv,
    scoring="f1"
)

print(nb_scores)

print("F1 medio:", nb_scores.mean())
print("Deviazione standard:", nb_scores.std())

[0.4119403  0.41260163 0.41028226 0.4231738  0.40144853 0.43377149
 0.42072858 0.39896907 0.41605839 0.41645244]
F1 medio: 0.4145426485315652
Deviazione standard: 0.00964955245927612


In [12]:
# Decision Tree
dt_scores = cross_val_score(
    DecisionTreeClassifier(random_state=42),
    X,
    y,
    cv=cv,
    scoring="f1"
)

print(dt_scores)

print("F1 medio:", dt_scores.mean())
print("Deviazione standard:", dt_scores.std())

[0.29639519 0.3221843  0.32802125 0.29589041 0.2866941  0.31929348
 0.3047619  0.27822581 0.33538672 0.3018617 ]
F1 medio: 0.30687148677662757
Deviazione standard: 0.017711030613464088
